In [14]:
import pandas as pd
import numpy as np
import random

# For reproducibility
random.seed(0)
np.random.seed(0)

In [15]:
ratings_df = pd.read_csv("../dataset/ratings.csv")
movies_df = pd.read_csv("../dataset/movies.csv")

In [16]:
# Randomly take 100,000 rows (without replacement)
ratings_sample = ratings_df.sample(n=100_000, random_state=0)
ratings_sample.head()

,userId,movieId,rating,timestamp
76998,594,519,3.0,836177816
13988377,90666,87430,2.0,1425221038
645617,4421,5502,3.0,1416152606
11081841,72074,2321,5.0,938652868
7789850,50602,7458,1.0,1419369317


In [20]:
# Map unique userId values to 0..(num_users-1)
user_map = {
    old_id: new_id for new_id, old_id in enumerate(ratings_sample["userId"].unique())
}

ratings_sample["user_idx"] = ratings_sample["userId"].map(user_map)

# Map unique movieId values to 0..(num_items-1)
item_map = {
    old_id: new_id for new_id, old_id in enumerate(ratings_sample["movieId"].unique())
}
ratings_sample["item_idx"] = ratings_sample["movieId"].map(item_map)


# Reverse mapping to retrieve movie titles from idx
reverse_item_map = {new: old for old, new in item_map.items()}

In [21]:
triplets = [
    (int(user), int(item), float(rating))
    for user, item, rating in zip(
        ratings_sample["user_idx"], ratings_sample["item_idx"], ratings_sample["rating"]
    )
]

# Show first few triplets
triplets[:100]

[(0, 0, 3.0),
 (1, 1, 2.0),
 (2, 2, 3.0),
 (3, 3, 5.0),
 (4, 4, 1.0),
 (5, 5, 3.5),
 (6, 6, 3.0),
 (7, 7, 5.0),
 (8, 8, 3.0),
 (9, 9, 2.5),
 (10, 10, 3.5),
 (11, 11, 3.5),
 (12, 12, 4.0),
 (13, 13, 3.5),
 (14, 14, 4.5),
 (15, 15, 4.0),
 (16, 16, 4.0),
 (17, 17, 3.5),
 (18, 18, 4.0),
 (19, 19, 4.5),
 (20, 20, 4.0),
 (21, 21, 1.0),
 (22, 22, 2.5),
 (23, 23, 4.0),
 (24, 24, 4.0),
 (25, 25, 5.0),
 (26, 26, 3.0),
 (27, 27, 3.0),
 (28, 28, 4.0),
 (29, 29, 3.0),
 (30, 30, 2.0),
 (31, 31, 4.0),
 (32, 32, 3.0),
 (33, 33, 4.0),
 (34, 34, 3.5),
 (35, 35, 1.0),
 (36, 36, 3.5),
 (37, 37, 4.0),
 (38, 38, 3.5),
 (39, 39, 3.0),
 (40, 40, 4.5),
 (41, 41, 2.0),
 (42, 42, 3.0),
 (43, 43, 2.0),
 (44, 44, 2.5),
 (45, 45, 4.0),
 (46, 46, 3.0),
 (47, 47, 3.0),
 (48, 48, 4.0),
 (49, 49, 5.0),
 (50, 50, 5.0),
 (51, 51, 4.5),
 (52, 52, 5.0),
 (53, 53, 3.0),
 (54, 54, 4.0),
 (55, 55, 4.5),
 (56, 56, 3.5),
 (57, 57, 4.0),
 (58, 58, 1.0),
 (59, 59, 1.0),
 (60, 60, 4.5),
 (61, 61, 4.0),
 (62, 43, 2.0),
 (63, 62, 3.

In [24]:
# Just to check sizes
num_users = ratings_sample["user_idx"].nunique()
num_items = ratings_sample["item_idx"].nunique()

print("Users:", num_users)
print("Items:", num_items)
print("Triplets:", len(triplets))

Users: 55079
Items: 10181
Triplets: 100000


In [57]:
from collections import defaultdict

# Shuffle and split into train/validation
triplets_shuffled = triplets.copy()
random.shuffle(triplets_shuffled)

split_idx = int(0.8 * len(triplets_shuffled))
train_triplets = triplets_shuffled[:split_idx]
val_triplets = triplets_shuffled[split_idx:]

print("Train size:", len(train_triplets))
print("Val size  :", len(val_triplets))

# which items a user has already rated
user_rated_items = defaultdict(set)
for u, i, r in train_triplets:
    user_rated_items[u].add(i)

# For building user content profiles later: list of (item_idx, rating) per user
user_ratings = defaultdict(list)
for u, i, r in train_triplets:
    user_ratings[u].append((i, r))

Train size: 80000
Val size  : 20000


In [ ]:
# Hyperparameters
num_factors = 20
learning_rate = 0.01
reg = 0.02
num_epochs = 10

# Global mean rating (from training data)
global_mean = np.mean([r for (_, _, r) in train_triplets])

# Parameters to be learned
user_bias = np.zeros(num_users, dtype=np.float32)
item_bias = np.zeros(num_items, dtype=np.float32)
user_factors = 0.1 * np.random.randn(num_users, num_factors).astype(np.float32)
item_factors = 0.1 * np.random.randn(num_items, num_factors).astype(np.float32)

Users: 55079
Items: 10181
Triplets: 100000


In [26]:
def svd_score(u, i):
    """
    Predict rating for internal user index u and item index i.
    """
    return (
        global_mean
        + user_bias[u]
        + item_bias[i]
        + np.dot(user_factors[u], item_factors[i])
    )


def rmse(triplets):
    se = 0.0
    for u, i, r in triplets:
        pred = svd_score(u, i)
        se += (r - pred) ** 2
    return np.sqrt(se / len(triplets))

In [58]:
def train_svd_sgd(train_triplets, val_triplets, num_epochs, learning_rate, reg):
    global user_bias, item_bias, user_factors, item_factors

    last_val_rmse = None

    for epoch in range(num_epochs):
        random.shuffle(train_triplets)

        squared_e_train = 0.0

        for u, i, r in train_triplets:
            # Current prediction & error
            pred = (
                global_mean
                + user_bias[u]
                + item_bias[i]
                + np.dot(user_factors[u], item_factors[i])
            )

            err = r - pred
            squared_e_train += err**2

            # Make copies to avoid overwriting while still using
            pu = user_factors[u].copy()
            qi = item_factors[i].copy()

            # Update biases
            user_bias[u] += learning_rate * (err - reg * user_bias[u])
            item_bias[i] += learning_rate * (err - reg * item_bias[i])

            # Update latent factors
            user_factors[u] += learning_rate * (err * qi - reg * pu)
            item_factors[i] += learning_rate * (err * pu - reg * qi)

        train_rmse = np.sqrt(squared_e_train / len(train_triplets))
        val_rmse = rmse(val_triplets)
        last_val_rmse = val_rmse  # remember last one

        # print(
        #     f"Epoch {epoch+1}/{num_epochs} "
        #     f"- Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}"
        # )

    return last_val_rmse

In [ ]:
def init_model(num_factors):
    global user_bias, item_bias, user_factors, item_factors

    user_bias = np.zeros(num_users, dtype=np.float32)
    item_bias = np.zeros(num_items, dtype=np.float32)
    user_factors = 0.1 * np.random.randn(num_users, num_factors).astype(np.float32)
    item_factors = 0.1 * np.random.randn(num_items, num_factors).astype(np.float32)


configs = [
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
    {"num_factors": 20, "lr": 0.015, "reg": 0.02, "epochs": 10},
]


results = []

for cfg in configs:
    print("\nTesting config:", cfg)

    # 1) Reinitialise model with this number of factors
    init_model(cfg["num_factors"])

    # 2) Train with this config's hyperparameters
    val_rmse = train_svd_sgd(
        train_triplets=train_triplets,
        val_triplets=val_triplets,
        num_epochs=cfg["epochs"],
        learning_rate=cfg["lr"],
        reg=cfg["reg"],
    )

    results.append((cfg, val_rmse))
    # print("Final val RMSE:", val_rmse)

print("\nSummary:")
for cfg, val_rmse in results:
    print(cfg, "-> val RMSE:", val_rmse)


Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 5}

Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 8}

Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 10}

Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 12}

Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 15}

Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 18}

Testing config: {'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 21}

Summary:
{'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 5} -> val RMSE: 0.9748526520578629
{'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 8} -> val RMSE: 0.968440659626425
{'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 10} -> val RMSE: 0.9675387061603773
{'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 12} -> val RMSE: 0.9676642488268435
{'num_factors': 20, 'lr': 0.015, 'reg': 0.02, 'epochs': 15} -> val RMSE: 0.9698353779892

In [70]:
# results from testing:
num_factors = 20
lr = 0.015
reg = 0.02
epochs = 10

In [71]:
# Reinitialise parameters if you change hyperparameters
user_bias = np.zeros(num_users, dtype=np.float32)
item_bias = np.zeros(num_items, dtype=np.float32)
user_factors = 0.1 * np.random.randn(num_users, num_factors).astype(np.float32)
item_factors = 0.1 * np.random.randn(num_items, num_factors).astype(np.float32)

train_svd_sgd(
    train_triplets,
    val_triplets,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    reg=reg,
)

print("Final validation RMSE:", rmse(val_triplets))

Final validation RMSE: 0.9714298885714634


In [66]:
def get_scores(u, top_n=10):
    # Items this user has already seen
    seen = user_rated_items[u]

    scores = []

    for i in range(num_items):
        if i in seen:
            continue  # skip already-rated items
        score = svd_score(u, i)
        scores.append((i, score))

    # Sort by predicted rating, descending
    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_n]


def recommend_cf_for_raw_user(raw_user_id, top_n=10):
    if raw_user_id not in user_map:
        print("User not in sampled ratings.")
        return

    u = user_map[raw_user_id]  # internal index

    scores = get_scores(raw_user_id)

    scores.sort(key=lambda x: x[1], reverse=True)
    top_items = scores[:top_n]

    print(f"CF-only recommendations for userId {raw_user_id}:\n")
    for item_idx, score in top_items:
        movie_id = reverse_item_map[item_idx]
        row = movies_df[movies_df["movieId"] == movie_id].iloc[0]
        print(f"{row['title']}  (predicted rating: {score:.2f})")


recommend_cf_for_raw_user(raw_user_id=1, top_n=5)

CF-only recommendations for userId 1:

Manchurian Candidate, The (1962)  (predicted rating: 4.42)
Seven Samurai (Shichinin no samurai) (1954)  (predicted rating: 4.37)
Rear Window (1954)  (predicted rating: 4.37)
Wallace & Gromit: A Close Shave (1995)  (predicted rating: 4.35)
Shawshank Redemption, The (1994)  (predicted rating: 4.35)


In [46]:
# Building CBF implementation
genome_scores = pd.read_csv("../dataset/genome-scores.csv")
genome_scores.head()

,movieId,tagId,relevance
0,1,1,0.02875
1,1,2,0.02375
2,1,3,0.06250
3,1,4,0.07575
4,1,5,0.14075


In [47]:
# Pivot to get a wide matrix: rows = movies, columns = tags
movie_feature_matrix = genome_scores.pivot(
    index="movieId", columns="tagId", values="relevance"
)

movie_feature_matrix.head()

tagId,1,2,3,4,5,6,7,8,9,10,...,1119,1120,1121,1122,1123,1124,1125,1126,1127,1128
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.02875,0.02375,0.06250,0.07575,0.14075,0.14675,0.06350,0.20375,0.2020,0.03075,...,0.04050,0.01425,0.03050,0.03500,0.14125,0.05775,0.03900,0.02975,0.08475,0.02200
2,0.04125,0.04050,0.06275,0.08275,0.09100,0.06125,0.06925,0.09600,0.0765,0.05250,...,0.05250,0.01575,0.01250,0.02000,0.12225,0.03275,0.02100,0.01100,0.10525,0.01975
3,0.04675,0.05550,0.02925,0.08700,0.04750,0.04775,0.04600,0.14275,0.0285,0.03875,...,0.06275,0.01950,0.02225,0.02300,0.12200,0.03475,0.01700,0.01800,0.09100,0.01775
4,0.03425,0.03800,0.04050,0.03100,0.06500,0.03575,0.02900,0.08650,0.0320,0.03150,...,0.05325,0.02800,0.01675,0.03875,0.18200,0.07050,0.01625,0.01425,0.08850,0.01500
5,0.04300,0.05325,0.03800,0.04100,0.05400,0.06725,0.02775,0.07650,0.0215,0.02975,...,0.05350,0.02050,0.01425,0.02550,0.19225,0.02675,0.01625,0.01300,0.08700,0.01600


In [48]:
# Number of items in your sampled dataset
num_items = ratings_sample["item_idx"].nunique()
num_tags = movie_feature_matrix.shape[1]
print("Number of tags:", num_tags)

# Create an empty matrix where each row will be the tag vector
aligned_movie_features = np.zeros((num_items, num_tags), dtype=np.float32)
genome_movie_ids = movie_feature_matrix.index

# Fill aligned_movie_features according to your internal index ordering
for original_movie_id, internal_idx in item_map.items():

    if original_movie_id in genome_movie_ids:

        tag_vector = movie_feature_matrix.loc[original_movie_id].values
        aligned_movie_features[internal_idx, :] = tag_vector

aligned_movie_features.shape

Number of tags: 1128


(10181, 1128)

In [49]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=True, with_std=True)
movie_features_scaled = scaler.fit_transform(aligned_movie_features)

In [50]:
from sklearn.decomposition import PCA

num_content_factors = 50

pca = PCA(n_components=num_content_factors, random_state=0)
movie_content_latent = pca.fit_transform(aligned_movie_features)

movie_content_latent.shape

(10181, 50)

In [51]:
# We want to build user profiles described as movie features from above

# For each user, store list of (item_idx, rating)
user_ratings = defaultdict(list)

for u, i, r in train_triplets:
    user_ratings[u].append((i, r))

len(user_ratings), list(user_ratings.items())[:1]  # quick peek

(47856, [(5686, [(2759, 0.5)])])

In [52]:
num_users = ratings_sample["user_idx"].nunique()
num_content_factors = movie_content_latent.shape[1]

user_content_profiles = np.zeros((num_users, num_content_factors), dtype=np.float32)

for u in range(num_users):
    rated = user_ratings.get(u, [])
    if not rated:
        # Cold-start user: leave as zero vector for now
        continue

    # Build weighted sum of movie content vectors
    num = np.zeros(num_content_factors, dtype=np.float32)
    denom = 0.0

    for i, r in rated:
        w = r - global_mean  # preference weight
        if w <= 0:
            # ignore movies rated below average by the user
            continue

        num += w * movie_content_latent[i]
        denom += abs(w)

    if denom > 0:
        user_content_profiles[u] = num / denom
    else:
        # fallback: simple average of all rated movies
        vecs = [movie_content_latent[i] for (i, r) in rated]
        user_content_profiles[u] = np.mean(vecs, axis=0)

In [53]:
from numpy.linalg import norm


def content_score(u, i):
    u_vec = user_content_profiles[u]
    i_vec = movie_content_latent[i]

    # cosine similiarity
    u_norm = norm(u_vec)
    i_norm = norm(i_vec)
    if u_norm == 0 or i_norm == 0:
        return 0

    return float(np.dot(u_vec, i_vec) / (u_norm * i_norm))

In [54]:
alpha = 0.7  # weight for CF vs content


def predict_hybrid(u, i, alpha=alpha):
    cf_part = svd_score(u, i)
    cb_part = content_score(u, i)
    return alpha * cf_part + (1 - alpha) * cb_part

In [55]:
# alpha * CF + (1 - alpha) * content


def rmse_hybrid(triplets, alpha):
    se = 0.0
    for u, i, r in triplets:
        cf_part = svd_score(u, i)
        cb_part = content_score(u, i)
        pred = alpha * cf_part + (1 - alpha) * cb_part
        se += (r - pred) ** 2
    return np.sqrt(se / len(triplets))

In [ ]:
alphas = [
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.91,
    0.92,
    0.93,
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
    1.0,
]  # 0 = pure content, 1 = pure CF

alpha_results = []

for a in alphas:
    val_rmse = rmse_hybrid(val_triplets, alpha=a)
    alpha_results.append((a, val_rmse))
    print(f"alpha = {a} -> val RMSE: {val_rmse:.4f}")

print("\nBest alpha:")
best_alpha, best_alpha_rmse = sorted(alpha_results, key=lambda x: x[1])[0]
print("alpha =", best_alpha, "with val RMSE =", best_alpha_rmse)

alpha = 0.7 -> val RMSE: 1.2906
alpha = 0.75 -> val RMSE: 1.1959
alpha = 0.8 -> val RMSE: 1.1139
alpha = 0.85 -> val RMSE: 1.0476
alpha = 0.9 -> val RMSE: 1.0001
alpha = 0.91 -> val RMSE: 0.9931
alpha = 0.92 -> val RMSE: 0.9870
alpha = 0.93 -> val RMSE: 0.9818
alpha = 0.94 -> val RMSE: 0.9775
alpha = 0.95 -> val RMSE: 0.9741
alpha = 0.96 -> val RMSE: 0.9717
alpha = 0.97 -> val RMSE: 0.9702
alpha = 0.98 -> val RMSE: 0.9697
alpha = 0.99 -> val RMSE: 0.9701
alpha = 1.0 -> val RMSE: 0.9714

Best alpha:
alpha = 0.98 with val RMSE = 0.9696670411924966
